# 06_ivf: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/06_ivf/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Trace a centroid assignment from training to retrieval.

Concepts: Trace a centroid assignment from training to retrieval.
Run: uv run python modules/06_ivf/examples/01_from_scratch.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

import numpy as np

from common.index_lessons import unit_catalog
from tinyvec.ivf import IVF


def main():
    _, matrix, query = unit_catalog(320)
    index = IVF(matrix, nlist=8, nprobe=1)
    distances = ((index.centers - query) ** 2).sum(axis=1)
    nearest = int(np.argmin(distances))
    candidates = index.candidates(query, 10)
    print("nearest centroid", nearest, "list sizes", [len(rows) for rows in index.lists])
    assert set(candidates) == set(index.lists[nearest])
    index.nprobe = 8
    assert set(index.candidates(query, 10)) == set(range(len(matrix)))


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/06_ivf/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra libs")
else:
    """Probe a trained FAISS inverted file.

    Concepts: Probe a trained FAISS inverted file.
    Run: uv run --extra libs python modules/06_ivf/examples/02_production.py
    Expected: Inspect the printed IDs or measurements and the checked invariant.
    CPU: seconds on the included small workload. Uses local data; no Docker or model download.
    """

    # course-extra: libs
    import faiss

    from common.index_lessons import unit_catalog

    def main():
        products, matrix, q = unit_catalog(640)
        faiss.omp_set_num_threads(1)
        coarse = faiss.IndexFlatIP(matrix.shape[1])
        index = faiss.IndexIVFFlat(coarse, matrix.shape[1], 8, faiss.METRIC_INNER_PRODUCT)
        index.train(matrix)
        index.add(matrix)
        index.nprobe = 3
        scores, rows = index.search(q[None, :], 10)
        print("occupancies", [index.invlists.list_size(i) for i in range(8)])
        print("candidate products", products.iloc[rows[0]].id.tolist(), "cosine scores", scores[0])
        assert index.is_trained and index.ntotal == len(matrix)

    if __name__ == "__main__":
        main()

## 03_measure

Source: `modules/06_ivf/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure the recall cost of probing fewer lists.

Concepts: Measure the recall cost of probing fewer lists.
Run: uv run python modules/06_ivf/examples/03_measure.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

from common.index_lessons import trial
from common.teaching import evidence


def main():
    observations = [trial("ivf", nlist=12, nprobe=p) for p in (1, 4, 12)]
    print(observations)
    evidence("lesson-06_ivf", {"observations": observations}, tier=2)


if __name__ == "__main__":
    main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.index_lessons import trial

print(trial("ivf", nlist=8, nprobe=1))

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.